In [299]:
import numpy as np
import pandas as pd
import jax
import jax.numpy as jnp
import optax
import unittest
from functools import partial 

import matplotlib.pyplot as plt

In [321]:
def loss_fn(z, Sigma_tilde):
    """
        z: vector supporting K(z)
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # defines jax-numpy loss
    d = Sigma_tilde.shape[0]
    K_z = get_K(z, d)
    log_det_term = jnp.log(jnp.linalg.det(K_z))
    trace_term = jnp.trace(jnp.matmul(K_z, Sigma_tilde))
    #print(-log_det_term + trace_term)
    return -log_det_term + trace_term

def get_K(z, d):
    """
    z: vector supporting precision matrix estimate K(z)
    d: dimension of Sigma
    """
    z_mat = z.reshape((d, d))
    
    # get lower tri part w/ the diagonal
    lower_tri_indices = np.tril_indices(d)
    z_vals = z_mat[lower_tri_indices]
    q = len(z_vals)
    
    E1 = np.zeros((d, q))
    E2 = np.zeros((d, q))
    
    for k, (i_k, j_k) in enumerate(zip(*lower_tri_indices)):
        E1[i_k, k] = 1
        E2[j_k, k] = 1

    diag_z = jnp.diag(z_vals)
    
    K = E1 @ diag_z @ E2.T + E2 @ diag_z @ E1.T
    
    return K

def initialize_z(Sigma_tilde):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """
    # Initialize z such that later K(z) can be defined as a diagonal matrix with positive entries
    z = np.zeros(Sigma_tilde.shape)
    diag_elements = np.maximum(0.01, 1.0 / np.diag(Sigma_tilde))
    diag_elements = diag_elements / 2  # avoid counting diagonal elements twice
    
    for i in range(len(diag_elements)):
        z[i, i] = diag_elements[i]

    # try adding off diag elements
    
    print("Sigma_tilde", Sigma_tilde)
    print("z", z)

    z = z.ravel()
        
    return z

def binary_search_pd_point(z, grad):
    """
        z: current z vector
        grad: current gradient of the loss function wrt z
        step_size: initial step size for the search
        Sigma_tilde: matrix of observed (privatized) covariance entries
    """ 
    lower, upper = 0.0, 1.0
    while upper - lower > 1e-6:
        # search for mid s.t. K(z - mid*grad) is in the pos def cone
        mid = (lower + upper) / 2
        z_new = z - mid * grad
        if not is_positive_definite(get_K(z_new)):
            # not sure about this yet
            upper = mid
        else:
            lower = mid
    return z - upper * grad
    
def optimize_lbfgs(Sigma_tilde, z, max_iters=100):
    """
        Sigma_tilde: matrix of observed (privatized) covariance entries
        z: vector supporting K(z)
    """

    d = Sigma_tilde.shape[0]
    
    # Linesearch, comment/uncomment the desired one
    # linesearch = optax.scale_by_backtracking_linesearch(max_backtracking_steps=15)
    linesearch = optax.scale_by_zoom_linesearch(max_linesearch_steps=15)
    
    # Optimizer
    opt = optax.chain(
        optax.lbfgs(),
        # Compare with or without linesearch by commenting this line
        linesearch,
    )
    
    # Initialize
    state = opt.init(z)
    
    # wrapper for the loss function that captures Sigma_tilde
    def loss_fn_wrapper(z):
        return loss_fn(z, Sigma_tilde)

    loss_fn_wrapper_v = partial(loss_fn, Sigma_tilde=Sigma_tilde)

    g_vals = []
    
    # Run optimization
    for i in range(max_iters):
        v, g = jax.value_and_grad(loss_fn_wrapper_v)(z)
        print(f"g {g}")
        print(f'Iteration: {i}, Value:{v:.2e}')
        
        updates, state = opt.update(g, state, z, value=v, grad=g, value_fn=loss_fn_wrapper_v)
        z = optax.apply_updates(z, updates)
        #z = jnp.add(z, updates)
        
        print(f"grad sum: {jnp.sum(jnp.abs(g))}")
        print(f"grad: {g}")
        print("z", z)
        
        K_z = get_K(z, d)
        assert is_positive_definite(K_z)
        log_det_1 = logdet_psd(K_z)
        s, log_det_2 = jnp.linalg.slogdet(K_z)
        print("compare log det", log_det_1, log_det_2)

        g_vals.append(jnp.sum(jnp.abs(g)))
        
    print(K_z)
    plt.plot(g_vals)
    plt.show()
    
    return z


def optimize_lbfgs_check(Sigma_tilde, z, max_iters=100, initial_step_size=1.0, step_size_reduction=0.5, min_step_size=1e-6):
    d = Sigma_tilde.shape[0]

    linesearch = optax.scale_by_zoom_linesearch(max_linesearch_steps=15)
    
    opt = optax.chain(
        optax.lbfgs(),
        linesearch,
    )

    state = opt.init(z)
    
    # Define a wrapper for the loss function that captures Sigma_tilde
    def loss_fn_wrapper(z):
        return loss_fn(z, Sigma_tilde)
    
    step_size = initial_step_size
    
    # optimization
    for i in range(max_iters):
        v, g = jax.value_and_grad(loss_fn_wrapper)(z)
        print(f'Iteration: {i}, Value:{v:.2e}')
        
        updates, state = opt.update(g, state, z, value=v, grad=g, value_fn=loss_fn_wrapper)
        
        # Backtracking loop
        while step_size >= min_step_size:
            z_new = optax.apply_updates(z, updates * step_size)
            K_z_new = get_K(z_new, d)
            
            if is_positive_definite(K_z_new):
                z = z_new
                break
            else:
                print("not positive definite, changing stepsize")
                step_size *= step_size_reduction
        
        if step_size < min_step_size:
            print("Minimum step size reached, but positive definiteness could not be ensured.")
            break
        
        print(f"grad: {jnp.sum(jnp.abs(g))}")
        print("z", z)
        K_z = get_K(z, d)
        log_det_1 = jnp.log(jnp.linalg.det(K_z))
        s, log_det_2 = jnp.linalg.slogdet(K_z)
        print("compare log det", log_det_1, log_det_2)

    print(K_z)
    return z


# helpers # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

def is_positive_definite(M):
    return jnp.all(jnp.linalg.eigvalsh(M) > 0)

def matrix_to_lower_triangular_dict(matrix):
    # cast z information as a mapping of lower triangular elements to their indices
    d = matrix.shape[0]  # Assuming square matrix
    lower_tri_dict = {}
    for row in range(d):
        for col in range(row + 1):
            if matrix[row, col]!=0:
                lower_tri_dict[(row, col)] = float(matrix[row, col])
    return lower_tri_dict

@jax.jit
def logdet_psd(A):
    # Compute the Cholesky decomposition
    L = jnp.linalg.cholesky(A)

    # Check if any entry of the Cholesky factor is NaN
    is_nan = jnp.any(jnp.isnan(L))

    def nan_case(_):
        return -jnp.inf

    def valid_case(L):
        diag_L = jnp.diag(L)
        log_det = 2 * jnp.sum(jnp.log(diag_L))
        return log_det

    # Use jax.cond to choose between the cases
    return jax.lax.cond(is_nan, nan_case, valid_case, L)

def generate_multivariate_normal_data(n, d):
    mean = np.zeros(d)
    
    # Define a random positive definite covariance matrix (4x4)
    A = np.random.rand(n, d)
    covariance_matrix = np.dot(A.transpose(), A)
    
    # Generate the multivariate normal dataset
    dataset = np.random.multivariate_normal(mean, covariance_matrix, n)
    
    return dataset, covariance_matrix


def get_Sigma_tilde_d4(cov):
    M = np.zeros_like(cov)
    
    # copy diagonal
    np.fill_diagonal(M, np.diag(cov))
    
    # copy the [0,3] and [3,0] elements
    M[0, 3] = cov[0, 3]
    M[3, 0] = cov[3, 0]
    
    return M

def is_positive_definite(K):
    """Check if a matrix is positive definite."""
    return jnp.all(jnp.linalg.eigvalsh(K) > 0)

# tests # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # # 

class TestAuxiliaryFunctions(unittest.TestCase):
    def test_K(self):
        d = 3
        z = np.array([[1/2., 0., 0.],
                      [0., 2/2., 0.],
                      [4., 0., 3/2.]]).flatten()
        K_result = get_K(z, d)
        K_true = np.array([[1., 0., 4.],
                           [0., 2., 0.],
                           [4., 0., 3.]])
        self.assertTrue(np.allclose(K_result, K_true, atol=1e-10))

    def test_initialize_z(self):
        Sigma_tilde = np.array([[1/1., 0., 1/4.],
                               [0., 1/2., 0.],
                               [1/4., 0., 1/3.]])
        d = Sigma_tilde.shape[0]
        
        z_mat = np.array([[1/2., 0., 0.],
                          [0., 2/2., 0.],
                          [4., 0., 3/2.]])  # in dict form, z_dict = {(0,0): 1/2, (1,1): 2/2, (2,2): 3/2, (2,0): 4}
        z_init = initialize_z(Sigma_tilde)
        z_init_mat = z_init.reshape((d, d))

        for i in range(d):
            self.assertAlmostEqual(z_init_mat[i,i], z_mat[i,i], places=10)

    def test_lower_triangular_dict(self):
        
        matrix = np.array([
            [1., 0., 4.],
            [0., 2., 0.],
            [4., 0., 3.]
        ])

        expected_dict = {(0, 0): 1.0, (1, 1): 2.0, (2, 0): 4.0, (2, 2): 3.0}

        result_dict = matrix_to_lower_triangular_dict(matrix)

        self.assertEqual(result_dict, expected_dict, "The resulting dictionary does not match the expected dictionary.")

In [ ]:
adult_path = "/Users/ceciliaferrando/Documents/UMASS/RESEARCH/DPsynthesisML/hd-datasets-master/clean/adult.csv"
adult_data_d4 = pd.read_csv(adult_path)[["age", "workclass", "fnlwgt", "education"]]

Sigma_true = adult_data_d4.cov().to_numpy()
Prec_true = np.linalg.inv(Sigma_true)
d = Sigma_true.shape[0]

Sigma_tilde = get_Sigma_tilde_d4(Sigma_true)
#Sigma_tilde = Sigma_tilde + np.random.normal(0, 0.01, size=Sigma_tilde.shape)
Prec_tilde = np.linalg.inv(Sigma_tilde)

print(f"Sigma_true, {Sigma_true}")
print(f"Prec_true, {Prec_true}")
print(f"Sigma_tilde, {Sigma_tilde}")
print(f"Prec_tilde, {Prec_tilde}")

z_init = initialize_z(Sigma_tilde)
K_z_init = get_K(z_init, d)
det_init = np.linalg.det(K_z_init)

z_opt = optimize_lbfgs(Sigma_tilde, z_init, max_iters=30)
#z_opt = optimize_lbfgs_check(Sigma_tilde, z_init, max_iters=100, initial_step_size=1.0, step_size_reduction=0.5, min_step_size=1e-6)


Sigma_true, [[26.54863137  1.12558197 -0.92236391 -0.29460695]
 [ 1.12558197  4.87413218 -0.10834532  0.20117863]
 [-0.92236391 -0.10834532  5.23528579 -0.19909199]
 [-0.29460695  0.20117863 -0.19909199 15.01169162]]
Prec_true, [[ 0.03827682 -0.00873201  0.00659933  0.00095573]
 [-0.00873201  0.20736026  0.00264208 -0.00291526]
 [ 0.00659933  0.00264208  0.19232948  0.00264487]
 [ 0.00095573 -0.00291526  0.00264487  0.06670765]]
Sigma_tilde, [[26.54863137  0.          0.         -0.29460695]
 [ 0.          4.87413218  0.          0.        ]
 [ 0.          0.          5.23528579  0.        ]
 [-0.29460695  0.          0.         15.01169162]]
Prec_tilde, [[0.03767493 0.         0.         0.00073938]
 [0.         0.20516473 0.         0.        ]
 [0.         0.         0.19101154 0.        ]
 [0.00073938 0.         0.         0.06662925]]
Sigma_tilde [[26.54863137  0.          0.         -0.29460695]
 [ 0.          4.87413218  0.          0.        ]
 [ 0.          0.          5.23528

/Users/ceciliaferrando/Documents/UMASS/RESEARCH/GaussianPGM/privateggm/lib/python3.10/site-packages/optax/_src/transform.py:1412: UserWarning: Explicitly requested dtype float64 requested in zeros is not available, and will be truncated to dtype float32. To enable more dtypes, set the jax_enable_x64 configuration option or the JAX_ENABLE_X64 shell environment variable. See https://github.com/google/jax#current-gotchas for more.
  lambda leaf: jnp.zeros((memory_size,) + leaf.shape, dtype=leaf.dtype),


grad sum: 0.5892177224159241
grad: [ 3.8146973e-06  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
  0.0000000e+00  0.0000000e+00  0.0000000e+00  0.0000000e+00
 -5.8921391e-01  0.0000000e+00  0.0000000e+00  0.0000000e+00]
z [0.01883336 0.         0.         0.         0.         0.10258237
 0.         0.         0.         0.         0.09550577 0.
 0.00073599 0.         0.         0.03330737]
compare log det -9.227386 -9.227386
g [-0.01147842  0.          0.          0.          0.          0.
  0.          0.          0.          0.          0.          0.
 -0.00244722  0.          0.         -0.00648117]
Iteration: 1, Value:1.32e+01


In [319]:
K_z_opt = get_K(z_opt, 4)

print("Sigma_true", Sigma_true)
print("Prec_true", Prec_true)
print("\n")
print("Sigma_tilde", Sigma_tilde)
print("Prec_tilde", Prec_tilde)
print("\n")
print("Sigma_opt", np.linalg.inv(K_z_opt))
print("Prec_opt", K_z_opt)



Sigma_true [[26.54863137  1.12558197 -0.92236391 -0.29460695]
 [ 1.12558197  4.87413218 -0.10834532  0.20117863]
 [-0.92236391 -0.10834532  5.23528579 -0.19909199]
 [-0.29460695  0.20117863 -0.19909199 15.01169162]]
Prec_true [[ 0.03827682 -0.00873201  0.00659933  0.00095573]
 [-0.00873201  0.20736026  0.00264208 -0.00291526]
 [ 0.00659933  0.00264208  0.19232948  0.00264487]
 [ 0.00095573 -0.00291526  0.00264487  0.06670765]]


Sigma_tilde [[26.54863137  0.          0.         -0.29460695]
 [ 0.          4.87413218  0.          0.        ]
 [ 0.          0.          5.23528579  0.        ]
 [-0.29460695  0.          0.         15.01169162]]
Prec_tilde [[0.03767493 0.         0.         0.00073938]
 [0.         0.20516473 0.         0.        ]
 [0.         0.         0.19101154 0.        ]
 [0.00073938 0.         0.         0.06662925]]


Sigma_opt [[26.548632    0.          0.         -0.29460692]
 [ 0.          4.874132    0.          0.        ]
 [ 0.          0.          5.2352858

In [320]:
unittest.main(argv=[''], verbosity=2, exit=False)

test_K (__main__.TestAuxiliaryFunctions) ... ok
test_initialize_z (__main__.TestAuxiliaryFunctions) ... ok
test_lower_triangular_dict (__main__.TestAuxiliaryFunctions) ... ok

----------------------------------------------------------------------
Ran 3 tests in 0.036s

OK


Sigma_tilde [[1.         0.         0.25      ]
 [0.         0.5        0.        ]
 [0.25       0.         0.33333333]]
z [[0.5 0.  0. ]
 [0.  1.  0. ]
 [0.  0.  1.5]]
